In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244080314

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp03
[学号] 20244080314
[机器指纹] 71540edf6c2a7b53
[RUN_ID] 20244080314-71540edf6c2a7b53
[Python] 3.11.17


实验1：正则表达式的基础匹配

In [2]:
import re

# \d+ / \s+ / \bcat\b
text = "cat on mat; later: cats and dogs"
print("1. 基础匹配")
print("数字 \\d+      :", re.findall(r"\d+", text))
print("空白 \\s+      :", re.findall(r"\s+", text))
print("单词 \\bcat\\b :", re.findall(r"\bcat\b", text))

# cat|dog 与交替顺序
print("\n2. 交替 cat|dog 与顺序差异")
text2 = "cat dog bird CAT"
print("cat|dog :", re.findall(r"cat|dog", text2))   # CAT 大小写不匹配
print('a|aa|aaa on "aaaa" :', re.findall(r"a|aa|aaa", "aaaa"))
print('aaa|aa|a on "aaaa" :', re.findall(r"aaa|aa|a", "aaaa"))

# 量词
print("\n3. 量词")
t = "ct cat caat caaat"
for p in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(p, re.findall(p, t))

1. 基础匹配
数字 \d+      : []
空白 \s+      : [' ', ' ', ' ', ' ', ' ', ' ']
单词 \bcat\b : ['cat']

2. 交替 cat|dog 与顺序差异
cat|dog : ['cat', 'dog']
a|aa|aaa on "aaaa" : ['a', 'a', 'a', 'a']
aaa|aa|a on "aaaa" : ['aaa', 'a']

3. 量词
ca*t ['ct', 'cat', 'caat', 'caaat']
ca+t ['cat', 'caat', 'caaat']
ca?t ['ct', 'cat']
ca{2}t ['caat']
ca{1,2}t ['cat', 'caat']


各量词的含义：
`*`：前面字符出现 **0 次或多次**。`ca*t` 中 `a` 可有可无，`ct`、`cat`、`caat`、`caaat` 均匹配。
`+`：前面字符出现 **1 次或多次**。`ca+t` 至少 1 个 `a`，故 `ct` 不匹配。
`?`：前面字符出现 **0 次或 1 次**。`ca?t` 至多 1 个 `a`，在 `caat`、`caaat` 中只匹配子串 `cat`。
`{2}`：前面字符**恰好出现 2 次**。`ca{2}t` 只匹配 `caat`。
`{1,2}`：前面字符出现 **1 到 2 次**。`ca{1,2}t` 匹配 `cat`、`caat`，不匹配 `caaat`。

实验2：分组、回溯引用与替换

In [3]:
import re

# 捕获组提取邮箱用户名和域名
print("1. 邮箱用户名与域名")
text_mail = "联系: alice@example.com; 备选: bob@test.org"
pattern_mail = r"(\w+)@(\w+\.\w+)"
print("findall 结果 :", re.findall(pattern_mail, text_mail))
for m in re.finditer(pattern_mail, text_mail):
    print(f"完整: {m.group(0)} | 用户名: {m.group(1)} | 域名: {m.group(2)}")

# 后向引用找连续重复单词
print("\n2. 连续重复单词")
text_dup = "cat cat, dog cat, bird bird"
dup = re.findall(r"\b(\w+)\s+\1\b", text_dup)
print("连续重复的单词 :", dup)

# 日期格式转换 
print("\n3. 日期 年-月-日 -> 月-日-年")
text_date = "2024-05-01\n2024/05/02\n2024.05.03"
pattern_date = r"(\d{4})([-/.])(\d{2})\2(\d{2})"
replaced = re.sub(pattern_date, r"\3-\4-\1", text_date)
print("转换前:")
print(text_date)
print("转换后:")
print(replaced)

# 电话号码统一
print("\n4. 电话号码统一格式")
text_phone = "联系我: 123-456-7890 或 987.654.3210"
pattern_phone = r"(\d{3})[-.](\d{3})[-.](\d{4})"
unified = re.sub(pattern_phone, r"(\1) \2-\3", text_phone)
print("转换前:", text_phone)
print("转换后:", unified)

1. 邮箱用户名与域名
findall 结果 : [('alice', 'example.com'), ('bob', 'test.org')]
完整: alice@example.com | 用户名: alice | 域名: example.com
完整: bob@test.org | 用户名: bob | 域名: test.org

2. 连续重复单词
连续重复的单词 : ['cat', 'bird']

3. 日期 年-月-日 -> 月-日-年
转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05-02-2024
05-03-2024

4. 电话号码统一格式
转换前: 联系我: 123-456-7890 或 987.654.3210
转换后: 联系我: (123) 456-7890 或 (987) 654-3210


实验3：使用 BeautifulSoup 解析 HTML

In [4]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲: 正则表达式</p>
<p class="course">第七讲: 结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

# 标题、h1
print("1. 标题与h1")
print("页面标题文本 :", soup.title.get_text())
h1 = soup.h1
print("h1 标签名    :", h1.name)
print("h1 文本      :", h1.get_text())

# find_all 段落与链接
print("\n2. class=course 段落与所有链接")
courses = soup.find_all("p", class_="course")
print("class=course 的段落:")
for p in courses:
    print(p.get_text())

links = soup.find_all("a")
print("所有链接:")
for a in links:
    print(f"文字: {a.get_text()} | href: {a.get('href')}")

# get_text 比较
print("\n3. get_text() 比较")
whole_text = soup.get_text()
print("整个文档 :", repr(whole_text))
p_text = soup.find("p", class_="course").get_text()
print("单个段落 :", repr(p_text))
a_text = soup.find("a").get_text()
print("单个链接 :", repr(a_text))

1. 标题与h1
页面标题文本 : 实验课程
h1 标签名    : h1
h1 文本      : 数据分析

2. class=course 段落与所有链接
class=course 的段落:
第六讲: 正则表达式
第七讲: 结构化数据
所有链接:
文字: 讲义六 | href: https://example.org/lecture06
文字: 讲义七 | href: https://example.org/lecture07

3. get_text() 比较
整个文档 : '\n实验课程\n数据分析\n第六讲: 正则表达式\n第七讲: 结构化数据\n讲义六\n讲义七\n\n'
单个段落 : '第六讲: 正则表达式'
单个链接 : '讲义六'
